# **Sub-daily FGS and Flood Warning and Alert Polling**

In [0]:
# =============================================================================
# master/master_polling.py
# Master notebook for the FGS polling cadence.
#
# SCHEDULE THIS NOTEBOOK -- not the individual job notebooks.
# Run every 30 minutes. The ingest job checks the time window itself and
# exits cleanly outside 10:00-22:00 UTC, so this schedule can run all day.
#
# HOW TO SCHEDULE IN DATABRICKS:
#   1. Open this notebook in the Databricks workspace
#   2. Click "Schedule" in the top right
#   3. Set to "Every 30 minutes"
#   4. Ensure the cluster is set to your DASH cluster
#   5. Save
#
# This notebook chains the individual job notebooks in sequence.
# Each job returns an exit value that this notebook reads to decide
# whether to continue to the next step.
# WHAT THIS RUN DOES
# ------------------
# Step 1: FGS ingest
#   Checks whether a new Flood Guidance Statement has been published since
#   the last run. Exits early outside 10:00-22:00 UTC or if no new FGS found.
#   Returns "new_data" if a statement was ingested, "no_change" otherwise.
#
# Step 2: FGS intersections
#   Only runs if step 1 returned "new_data".
#   Intersects the new FGS polygons against EA flood areas and constituencies.
#   Timeout: 10 minutes.
#
# Step 3: Flood warnings ingest
#   Runs unconditionally on every poll cycle.
#   Fetches the current state of all EA flood warnings and alerts from the
#   EA Real-Time Flood Monitoring API and writes to two Delta tables:
#     - flood_warnings_current  (overwritten each run -- live dashboard)
#     - flood_warnings_log      (append-only event log -- audit / verification)
#   Returns "success" if warnings were processed, "no_change" if the API
#   returned an empty feed (unusual -- typically means no active events).
#   Timeout: 5 minutes.
#
# NOTE ON POLLING FREQUENCY VS. API REFRESH RATE
# ------------------------------------------------
# The EA flood warnings API refreshes every 15 minutes. Running step 3
# hourly means we may miss up to three intermediate state changes between
# polls. In practice this is acceptable for the log -- severity transitions
# and CLOSED events are captured at the next poll regardless. The current
# state table is always accurate at the moment of the run.
# If DASH ever supports sub-hourly scheduling, tightening to 30 minutes
# (cron: 0 0/30 * * * ?) would give better event resolution during fast-
# moving flood situations.
#
# STEP DEPENDENCIES
# -----------------
# Step 2 depends on step 1 (only runs on new FGS data).
# Step 3 has no dependencies -- it always runs regardless of FGS outcome.
# A failure in step 1 or 2 does NOT prevent step 3 from running (see below).
#
# FAILURE HANDLING
# ----------------
# Step 1 failure: raises immediately -- step 2 and step 3 do not run.
# Step 2 failure: raises immediately -- step 3 does not run.
# Step 3 failure: raises after steps 1 and 2 have completed.
#
# If you want step 3 to run even when steps 1/2 fail, wrap steps 1/2 in a
# try/except and proceed to step 3 regardless. This is intentionally not
# done here to keep the notebook simple and failure modes obvious.
#
# Job numbering:
#   01_ingest_fgs               -- FGS ingest from FFC API
#   07_compute_intersections    -- FGS polygon intersections
#   11_flood_warnings_ingest    -- EA flood warnings current state + log

# =============================================================================

import sys
sys.path.insert(0, "/Workspace/Users/jon.payne@environment-agency.gov.uk/FGS_Notebooks")   # Adjust to your repo path




In [0]:
# =============================================================================
# STEP 1: INGEST FGS
# Poll the FFC API and write any new FGS to Delta.
# Exit values:
#   "success"        -- new FGS found and written, proceed to intersections
#   "no_change"      -- nothing new since last poll, stop here
#   "outside_window" -- outside 10:00-22:00 UTC polling window, stop here
#   "no_data"        -- API returned no statements (unusual), stop here
# =============================================================================

print("=" * 60)
print("STEP 1: Ingest FGS")
print("=" * 60)

ingest_result = dbutils.notebook.run(
    path      = "/Workspace/Users/jon.payne@environment-agency.gov.uk/FGS_Notebooks/jobs/01_ingest_fgs",
    timeout_seconds = 300,    # 5 minutes -- fail if ingest takes longer
    arguments = {}
)

print(f"Ingest result: {ingest_result}")

# Only proceed to intersection computation if ingest found new data.
# All other exit values mean there is nothing new to intersect.
if ingest_result != "success":
    print(f"Ingest exited with '{ingest_result}'. No intersection run needed.")
    dbutils.notebook.exit(ingest_result)




In [0]:
# =============================================================================
# STEP 2: COMPUTE INTERSECTIONS
# Intersect the newly ingested FGS polygons against EA flood areas
# and parliamentary constituencies.
# Only runs if step 1 returned "success".
# =============================================================================

print("=" * 60)
print("STEP 2: Compute intersections")
print("=" * 60)

intersect_result = dbutils.notebook.run(
    path      = "/Workspace/Users/jon.payne@environment-agency.gov.uk/FGS_Notebooks/jobs/07_compute_intersections",
    timeout_seconds = 600,    # 10 minutes
    arguments = {"recompute_all": "false"}   # Only process the new statement
)

print(f"Intersection result: {intersect_result}")

print("Polling run complete.")
#dbutils.notebook.exit("success")


In [0]:
# =============================================================================
# STEP 3: FLOOD WARNINGS INGEST
# Runs unconditionally. Fetches current EA flood warnings and alerts.
# =============================================================================

print("=" * 60)
print("STEP 3: Flood warnings ingest")
print("=" * 60)

warnings_result = dbutils.notebook.run(
    path            = "/Workspace/Users/jon.payne@environment-agency.gov.uk/FGS_Notebooks/jobs/11_flood_warnings_ingest",
    timeout_seconds = 300,    # 5 minutes
    arguments       = {}
)

print(f"Flood warnings result: {warnings_result}")

if warnings_result not in ("success", "no_change"):
    raise Exception(f"Flood warnings ingest returned unexpected value: {warnings_result}")


In [0]:
# =============================================================================
# COMPLETE
# =============================================================================

print("=" * 60)
print("Polling run complete.")
print(f"  FGS ingest:            {ingest_result}")
print(f"  FGS intersections:     {'ran' if ingest_result == 'new_data' else 'skipped'}")
print(f"  Flood warnings ingest: {warnings_result}")
print("=" * 60)

dbutils.notebook.exit("success")
